# Pitch Vision — Stage 4 (Analytics Layer)

This is the real Build Plan **Stage 4** (not the tactics/topview notebook, which is actually Stage 7 despite its filename) — the first time this pipeline runs on the **side-view/broadcast model** (`best_broadcast.pt`) instead of the topview drone model. Everything through Stage 3 (tracking, team colors, possession, camera motion, homography, speed/distance) gets rebuilt here for THIS camera angle, then two new pieces get built on top:

- **4.1 Position heatmaps** — where each player spent the match, as a density grid (exported as JSON, not a baked image, so a future frontend can render it interactively).
- **4.2 Custom player-rating model** — an explainable 1–10 rating per player from distance covered, sprint count, possession involvement, and ball recoveries. Deliberately not a pass/dribble-event model (4.3, the optional stretch phase) — passes/dribbles are noisier heuristics on 2D tracking and are named as future work rather than built.

**Why this needs its own calibration, separate from the topview notebook:** the pixel-to-pitch homography (Stage 3.2) is specific to one camera's exact framing. A broadcast/side-view clip is a different camera angle entirely, so its 4 pitch-corner pixel coordinates have to be read off THIS clip's own first frame — that's not a bug to route around, it's what a fixed calibration means. Broadcast footage also typically pans/zooms, unlike the static drone shot, so the camera-motion compensation step (Stage 3.1) actually does real work here instead of reading near-zero.

**Before running:** Runtime → Change runtime type → GPU (T4 is enough).

In [ ]:
!pip install -q ultralytics supervision
import os
for d in ['utils', 'trackers', 'team_assigner', 'player_ball_assigner',
          'camera_movement_estimator', 'view_transformer', 'speed_and_distance_estimator',
          'heatmap', 'player_rating']:
    os.makedirs(d, exist_ok=True)
print("Project folders created.")

In [ ]:
import torch
if torch.cuda.is_available():
    print(f"GPU detected: {torch.cuda.get_device_name(0)} -- inference will run fast.")
else:
    print("*** NO GPU DETECTED -- inference will silently run on CPU instead. ***")
    print("This will NOT error out -- it'll just take 20-30x longer (a ~1-2 minute detection")
    print("pass becomes ~20+ minutes) with no warning until you notice the clock.")
    print("Fix: Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4), then")
    print("Runtime -> Restart session, and re-run from the top.")

In [ ]:
from google.colab import files
print("Upload best_broadcast.pt (your Stage 1.1 side-view/broadcast fine-tuned model):")
uploaded = files.upload()
model_path = list(uploaded.keys())[0]
print("Using model:", model_path)

In [ ]:
print("Upload a broadcast/side-view clip (from your project's input_videos/ folder --")
print("licensed footage, e.g. the DFL Bundesliga Data Shootout dataset mentioned in the Build Plan):")
uploaded_clip = files.upload()
clip_path = list(uploaded_clip.keys())[0]
print("Using clip:", clip_path)

## Writing the pipeline modules

In [ ]:
%%writefile utils/__init__.py


In [ ]:
%%writefile trackers/__init__.py


In [ ]:
%%writefile team_assigner/__init__.py


In [ ]:
%%writefile player_ball_assigner/__init__.py


In [ ]:
%%writefile camera_movement_estimator/__init__.py


In [ ]:
%%writefile view_transformer/__init__.py


In [ ]:
%%writefile speed_and_distance_estimator/__init__.py


In [ ]:
%%writefile heatmap/__init__.py


In [ ]:
%%writefile player_rating/__init__.py


In [ ]:
%%writefile utils/bbox_utils.py
def get_center_of_bbox(bbox):
    x1, y1, x2, y2 = bbox
    return int((x1 + x2) / 2), int((y1 + y2) / 2)


def get_bbox_width(bbox):
    return bbox[2] - bbox[0]


def get_foot_position(bbox):
    x1, y1, x2, y2 = bbox
    return int((x1 + x2) / 2), int(y2)


def measure_distance(p1, p2):
    return ((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2) ** 0.5


In [ ]:
%%writefile utils/video_utils.py
import cv2


def read_video(video_path, target_width=1920):
    # Resize down while reading, not after — our clip is native 4K (3840x2160), and
    # holding all ~360 frames in memory at full 4K is ~9GB on its own, enough to crash
    # Colab's free-tier RAM by itself. target_width=1920 matches the resolution our
    # training data was stored at, so detection stays consistent with training too.
    cap = cv2.VideoCapture(video_path)
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        h, w = frame.shape[:2]
        if target_width is not None and w > target_width:
            scale = target_width / w
            frame = cv2.resize(frame, (target_width, int(h * scale)), interpolation=cv2.INTER_AREA)
        frames.append(frame)
    cap.release()
    return frames


def get_native_frame(video_path, frame_idx):
    """
    Reads exactly ONE frame directly from the source video file at its original,
    un-downsampled resolution (e.g. native 4K even though read_video() gives back
    1920-wide frames for detection/tracking/drawing).

    Used only for team-color sampling: a player crop that's already tiny gets made
    even blurrier by the resize read_video() does to keep RAM usage sane, which
    contaminates shirt-color sampling with blended-in grass pixels. Re-reading just
    the handful of frames we actually need color from, at full detail, avoids that
    without holding the whole clip in memory at 4K.
    """
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        raise RuntimeError(f"Could not read frame {frame_idx} from {video_path}")
    return frame


def save_video(output_video_frames, output_video_path, fps=25):
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    h, w = output_video_frames[0].shape[:2]
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (w, h))
    for frame in output_video_frames:
        out.write(frame)
    out.release()


def render_video_streaming(video_frames, output_path, draw_fns, fps=25):
    """
    Renders an annotated video WITHOUT ever holding a second full copy of the clip in
    memory: draws each frame, writes it straight to the video file, then moves on --
    one frame at a time.

    Prefer this over the pattern `save_video(some_drawer.draw_x(some_drawer2.draw_y(video_frames, ...), ...))`
    for any clip longer than a couple hundred frames. Each list-based draw_* method
    (Tracker.draw_annotations, SpeedAndDistanceEstimator.draw_speed_and_distance, ...)
    builds and returns a FULL new copy of the clip; chaining N of them, plus the
    original video_frames still being referenced, means roughly (N+1) full copies of
    every frame in the clip alive in RAM at the same time. That's exactly what
    crashes a Colab session with "used all available RAM" on a longer clip -- even
    when detection/tracking (the actual GPU-heavy step) already completed fine.

    video_frames: the original frames -- mutated in place, frame by frame, as they're
    drawn on and written out. Don't rely on them being unmodified afterward.
    draw_fns: list of callables, each `(frame, frame_num) -> frame`, applied in order
    to every frame before it's written -- e.g.
    [lambda f, n: tracker.draw_frame_annotations(f, n, tracks, team_ball_control),
     lambda f, n: speed_estimator.draw_frame_speed_and_distance(f, n, tracks)]
    """
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    h, w = video_frames[0].shape[:2]
    out = cv2.VideoWriter(output_path, fourcc, fps, (w, h))
    for frame_num, frame in enumerate(video_frames):
        for draw_fn in draw_fns:
            frame = draw_fn(frame, frame_num)
        out.write(frame)
    out.release()


In [ ]:
%%writefile trackers/tracker.py
import os
import pickle
import sys

import cv2
import numpy as np
import supervision as sv
from ultralytics import YOLO

sys.path.append('/content')
from utils.bbox_utils import get_center_of_bbox, get_bbox_width


class Tracker:
    def __init__(self, model_path):
        self.model = YOLO(model_path)
        self.tracker = sv.ByteTrack()

    def detect_frames(self, frames, conf=0.2, imgsz=1280, batch_size=20):
        # imgsz=1280 matches training — inference at the default 640 would shrink the
        # ball back down below what the model was actually trained to recognize.
        detections = []
        for i in range(0, len(frames), batch_size):
            batch = self.model.predict(frames[i:i + batch_size], conf=conf, imgsz=imgsz, verbose=False)
            detections += batch
        return detections

    def get_object_tracks(self, frames, read_from_stub=False, stub_path=None):
        if read_from_stub and stub_path is not None and os.path.exists(stub_path):
            with open(stub_path, 'rb') as f:
                return pickle.load(f)

        detections = self.detect_frames(frames)

        tracks = {"players": [], "ball": []}

        for frame_num, detection in enumerate(detections):
            cls_names = detection.names  # {0: 'player', 1: 'ball'}
            cls_names_inv = {v: k for k, v in cls_names.items()}

            detection_supervision = sv.Detections.from_ultralytics(detection)
            detection_with_tracks = self.tracker.update_with_detections(detection_supervision)

            tracks["players"].append({})
            tracks["ball"].append({})

            # players get persistent track_ids from ByteTrack
            for frame_detection in detection_with_tracks:
                bbox = frame_detection[0].tolist()
                cls_id = frame_detection[3]
                track_id = frame_detection[4]
                if cls_id == cls_names_inv.get('player'):
                    tracks["players"][frame_num][track_id] = {"bbox": bbox}

            # the ball gets a hardcoded track_id of 1 — there's only ever one, no need to
            # track its identity across frames, just its position
            for frame_detection in detection_supervision:
                bbox = frame_detection[0].tolist()
                cls_id = frame_detection[3]
                if cls_id == cls_names_inv.get('ball'):
                    tracks["ball"][frame_num][1] = {"bbox": bbox}

        if stub_path is not None:
            with open(stub_path, 'wb') as f:
                pickle.dump(tracks, f)

        return tracks

    def draw_ellipse(self, frame, bbox, color, track_id=None):
        y2 = int(bbox[3])
        x_center, _ = get_center_of_bbox(bbox)
        width = get_bbox_width(bbox)

        cv2.ellipse(
            frame,
            center=(x_center, y2),
            axes=(int(width), int(0.35 * width)),
            angle=0.0,
            startAngle=-45,
            endAngle=235,
            color=color,
            thickness=2,
            lineType=cv2.LINE_4,
        )

        rect_w, rect_h = 40, 20
        x1_rect = x_center - rect_w // 2
        x2_rect = x_center + rect_w // 2
        y1_rect = (y2 - rect_h // 2) + 15
        y2_rect = (y2 + rect_h // 2) + 15

        if track_id is not None:
            cv2.rectangle(frame, (int(x1_rect), int(y1_rect)), (int(x2_rect), int(y2_rect)), color, cv2.FILLED)
            x1_text = x1_rect + 12
            if track_id > 99:
                x1_text -= 10
            cv2.putText(frame, f"{track_id}", (int(x1_text), int(y1_rect + 15)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)

        return frame

    def draw_triangle(self, frame, bbox, color):
        y = int(bbox[1])
        x, _ = get_center_of_bbox(bbox)
        points = np.array([[x, y], [x - 10, y - 20], [x + 10, y - 20]])
        cv2.drawContours(frame, [points], 0, color, cv2.FILLED)
        cv2.drawContours(frame, [points], 0, (0, 0, 0), 2)
        return frame

    def draw_frame_annotations(self, frame, frame_num, tracks, team_ball_control):
        """
        Single-frame version of draw_annotations -- draws directly onto `frame` (the
        caller owns whether that's safe to mutate) and returns it. This is what the
        memory-safe streaming renderer (utils/video_utils.py's render_video_streaming)
        calls per frame, instead of ever building a second full copy of the clip.
        """
        player_dict = tracks["players"][frame_num]
        ball_dict = tracks["ball"][frame_num]

        for track_id, player in player_dict.items():
            color = player.get("team_color", (0, 0, 255))
            frame = self.draw_ellipse(frame, player["bbox"], color, track_id)
            if player.get("has_ball", False):
                frame = self.draw_triangle(frame, player["bbox"], (0, 0, 255))

        for _, ball in ball_dict.items():
            frame = self.draw_triangle(frame, ball["bbox"], (0, 255, 0))

        if len(team_ball_control) > 0 and frame_num < len(team_ball_control):
            so_far = team_ball_control[:frame_num + 1]
            t1 = int((so_far == 1).sum())
            t2 = int((so_far == 2).sum())
            total = t1 + t2
            if total > 0:
                cv2.putText(frame, f"Team 1 Ball Control: {t1 / total * 100:.1f}%", (50, 50),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)
                cv2.putText(frame, f"Team 2 Ball Control: {t2 / total * 100:.1f}%", (50, 90),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)

        return frame

    def draw_annotations(self, video_frames, tracks, team_ball_control):
        """
        List-based version -- builds and returns a full second copy of the clip.
        Fine for a short clip; for anything more than a couple hundred frames, prefer
        render_video_streaming() in utils/video_utils.py with draw_frame_annotations,
        which writes each frame straight to disk instead of holding it. Kept here for
        backward compatibility (and it's what draw_frame_annotations now factors out of).
        """
        output_frames = []
        for frame_num, frame in enumerate(video_frames):
            frame = frame.copy()
            frame = self.draw_frame_annotations(frame, frame_num, tracks, team_ball_control)
            output_frames.append(frame)
        return output_frames


In [ ]:
%%writefile team_assigner/team_assigner.py
import cv2
import numpy as np
from sklearn.cluster import KMeans


class TeamAssigner:
    """
    Assigns each tracked person to a team by clustering shirt color, and — since our
    detector only has a 'player' class (no separate 'referee' class in training data) —
    flags anyone whose shirt color doesn't cleanly match either team cluster as a
    non-player (referee) instead of forcing them into the nearest team.

    History of what didn't work, kept here because the next person touching this file
    (possibly future-me) will otherwise re-try the same dead ends:
      1. Top-half-of-box sampling (broadcast/side-view assumption: shirt on top, shorts
         below) -- wrong for an overhead view, where the top of a tiny box is head/hair.
      2. Full-box + inner 2-cluster KMeans, treating corner pixels as "background" --
         fails on tight boxes where the corners are still the player's own body.
      3. Plain median of the whole box, even the whole native-resolution box -- still
         failed in practice. Measured real output looked like BGR (91, 133, 109) and
         (101, 145, 124) for the two teams: nearly identical AND both green-dominant
         (G channel highest in both) -- a dead giveaway that grass pixels, not shirt
         pixels, were winning the median vote. A generously-sized/loosely-fit detection
         box around a small player can be majority background even when "tight" by eye.
    """

    # Grass in this footage (real turf, mowing stripes) sits in a fairly consistent
    # green hue band regardless of light/dark stripe -- stripes differ in brightness
    # (V), not hue. OpenCV hue is 0-179. Saturation gate avoids excluding dark/desaturated
    # shirts that merely happen to fall in the same hue range.
    GRASS_HUE_LOW = 25
    GRASS_HUE_HIGH = 95
    GRASS_SAT_MIN = 40

    def __init__(self):
        self.team_colors = {}
        self.player_team_dict = {}
        self.kmeans = None
        self.outlier_threshold = None

    def get_player_color(self, frame, bbox):
        x1, y1, x2, y2 = [int(v) for v in bbox]
        image = frame[y1:y2, x1:x2]
        if image.size == 0:
            return np.array([0, 0, 0])

        # Trim a small margin off each edge -- the outermost pixels of even a tight box
        # are the most likely to be anti-aliased/motion-blurred blends with whatever's
        # just outside the player.
        h, w = image.shape[:2]
        my, mx = int(h * 0.1), int(w * 0.1)
        if h - 2 * my > 0 and w - 2 * mx > 0:
            image = image[my:h - my, mx:w - mx]

        # Explicitly drop grass-hued pixels before summarizing color. This is a more
        # direct fix than just hoping a tighter crop avoids background: it targets the
        # actual contamination (green pitch) by color, so it still works even when the
        # detection box itself is loose or the player is tiny and blurry.
        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
        hue, sat = hsv[:, :, 0], hsv[:, :, 1]
        is_grass = (hue >= self.GRASS_HUE_LOW) & (hue <= self.GRASS_HUE_HIGH) & (sat >= self.GRASS_SAT_MIN)

        pixels = image.reshape(-1, 3)
        keep = ~is_grass.reshape(-1)
        kept_pixels = pixels[keep]

        # If almost everything got excluded (box is nearly all pitch -- heavy occlusion,
        # a bad box, or a genuinely green/olive kit), fall back to the full crop rather
        # than return a median of a handful of pixels.
        if len(kept_pixels) < 0.15 * len(pixels):
            kept_pixels = pixels

        return np.median(kept_pixels, axis=0)

    def assign_team_colors_from_samples(self, player_colors_dict):
        """
        Lower-level entry point: takes a pre-computed {track_id: color} mapping (e.g.
        each player's color already aggregated/medianed across several frames by the
        caller) and does the actual team clustering + referee-outlier detection.
        Separated from assign_team_colors() so the pipeline can aggregate samples across
        multiple frames for stability instead of trusting a single frame.
        """
        track_ids = list(player_colors_dict.keys())
        player_colors = np.array([player_colors_dict[tid] for tid in track_ids])

        kmeans = KMeans(n_clusters=2, init="k-means++", n_init=10)
        kmeans.fit(player_colors)
        self.kmeans = kmeans

        self.team_colors[1] = kmeans.cluster_centers_[0]
        self.team_colors[2] = kmeans.cluster_centers_[1]

        # distance from each person's color to their nearest team-cluster center — a
        # referee's kit color won't match either team well, so this distance spikes for
        # them specifically. Threshold is data-driven (mean + 2*std), not a hardcoded guess.
        distances = []
        for color in player_colors:
            label = kmeans.predict(color.reshape(1, -1))[0]
            distances.append(np.linalg.norm(color - kmeans.cluster_centers_[label]))
        distances = np.array(distances)
        self.outlier_threshold = distances.mean() + 2 * distances.std() if len(distances) > 1 else np.inf

        for track_id, color in zip(track_ids, player_colors):
            label = kmeans.predict(color.reshape(1, -1))[0]
            dist = np.linalg.norm(color - kmeans.cluster_centers_[label])
            self.player_team_dict[track_id] = "referee" if dist > self.outlier_threshold else int(label) + 1

    def assign_team_colors(self, frame, player_detections):
        """Single-frame convenience wrapper around assign_team_colors_from_samples()."""
        player_colors = {
            track_id: self.get_player_color(frame, detection["bbox"])
            for track_id, detection in player_detections.items()
        }
        self.assign_team_colors_from_samples(player_colors)

    def get_player_team(self, frame, player_bbox, player_id):
        if player_id in self.player_team_dict:
            return self.player_team_dict[player_id]

        color = self.get_player_color(frame, player_bbox)
        label = self.kmeans.predict(color.reshape(1, -1))[0]
        dist = np.linalg.norm(color - self.kmeans.cluster_centers_[label])

        team = "referee" if (self.outlier_threshold is not None and dist > self.outlier_threshold) else int(label) + 1
        self.player_team_dict[player_id] = team
        return team


In [ ]:
%%writefile player_ball_assigner/player_ball_assigner.py
import sys

sys.path.append('/content')
from utils.bbox_utils import get_center_of_bbox


class PlayerBallAssigner:
    def __init__(self, max_player_ball_distance=70):
        self.max_player_ball_distance = max_player_ball_distance

    def assign_ball_to_player(self, players, ball_bbox):
        ball_x, ball_y = get_center_of_bbox(ball_bbox)
        minimum_distance = float("inf")
        assigned_player = -1

        for player_id, player in players.items():
            bbox = player["bbox"]
            distance_left = ((bbox[0] - ball_x) ** 2 + (bbox[-1] - ball_y) ** 2) ** 0.5
            distance_right = ((bbox[2] - ball_x) ** 2 + (bbox[-1] - ball_y) ** 2) ** 0.5
            distance = min(distance_left, distance_right)

            if distance < self.max_player_ball_distance and distance < minimum_distance:
                minimum_distance = distance
                assigned_player = player_id

        return assigned_player


In [ ]:
%%writefile camera_movement_estimator/camera_movement_estimator.py
import sys
sys.path.append('/content')

import cv2
import numpy as np

from utils.bbox_utils import measure_distance


class CameraMovementEstimator:
    """
    Estimates how much the CAMERA itself moved between consecutive frames (pan/tilt/
    drift), using sparse optical flow tracked on background features -- not players --
    so that "how far a player moved" can later be separated from "how far the camera
    moved and dragged everything in the frame along with it."

    On this project's footage the camera is a fixed, static overhead rig (the whole
    pitch stays framed identically across the whole clip), so in practice this should
    come out close to [0, 0] every frame. It's still worth running rather than assuming
    that: if a future clip comes from a genuinely panning or drone-drifting camera,
    speed/distance numbers would otherwise silently include the camera's own motion.
    """

    def __init__(self, first_frame):
        self.min_distance = 5

        # Only look for trackable features in narrow strips along the very top and
        # bottom of the frame. In a full-pitch overhead shot, players are rarely up
        # against those edges, so these strips are much more likely to be genuine
        # static background (stadium structure, advertising boards, empty grass)
        # whose only frame-to-frame motion is the camera's own.
        first_gray = cv2.cvtColor(first_frame, cv2.COLOR_BGR2GRAY)
        h, w = first_gray.shape
        mask_features = np.zeros_like(first_gray)
        mask_features[0:int(h * 0.08), :] = 1
        mask_features[int(h * 0.92):h, :] = 1

        self.feature_params = dict(maxCorners=100, qualityLevel=0.3, minDistance=3, blockSize=7, mask=mask_features)
        self.lk_params = dict(
            winSize=(15, 15),
            maxLevel=2,
            criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03),
        )

    def get_camera_movement(self, frames):
        """Returns one [dx, dy] per frame: how far the camera moved since the PREVIOUS
        frame (pixels). Frame 0 is always [0, 0] (nothing to compare it to)."""
        camera_movement = [[0, 0] for _ in range(len(frames))]

        old_gray = cv2.cvtColor(frames[0], cv2.COLOR_BGR2GRAY)
        old_features = cv2.goodFeaturesToTrack(old_gray, **self.feature_params)

        for frame_num in range(1, len(frames)):
            new_gray = cv2.cvtColor(frames[frame_num], cv2.COLOR_BGR2GRAY)

            if old_features is None or len(old_features) == 0:
                old_features = cv2.goodFeaturesToTrack(old_gray, **self.feature_params)
                if old_features is None:
                    old_gray = new_gray
                    continue

            new_features, status, _ = cv2.calcOpticalFlowPyrLK(old_gray, new_gray, old_features, None, **self.lk_params)

            if new_features is None or status is None:
                old_gray = new_gray
                old_features = cv2.goodFeaturesToTrack(old_gray, **self.feature_params)
                continue

            status = status.flatten()
            good_new = new_features[status == 1]
            good_old = old_features[status == 1]

            # take the single largest-displacement feature as the camera's movement
            # estimate, not an average -- an average gets pulled toward zero by
            # features that happen to sit on something that moved independently
            # (e.g. a player who wandered into the tracked strip), while the true
            # background motion is consistent and shows up as the dominant shift.
            max_distance = 0.0
            camera_movement_x, camera_movement_y = 0.0, 0.0
            for new, old in zip(good_new, good_old):
                new_pt, old_pt = new.ravel(), old.ravel()
                distance = measure_distance(new_pt, old_pt)
                if distance > max_distance:
                    max_distance = distance
                    camera_movement_x = old_pt[0] - new_pt[0]
                    camera_movement_y = old_pt[1] - new_pt[1]

            if max_distance > self.min_distance:
                camera_movement[frame_num] = [camera_movement_x, camera_movement_y]
                old_features = cv2.goodFeaturesToTrack(new_gray, **self.feature_params)
            else:
                old_features = good_new.reshape(-1, 1, 2) if len(good_new) > 0 else cv2.goodFeaturesToTrack(new_gray, **self.feature_params)

            old_gray = new_gray

        return camera_movement

    def adjust_positions_to_tracks(self, tracks, camera_movement):
        """
        Subtracts the camera's own movement (accumulated from frame 0) from every
        tracked player/ball bbox, writing the result as "adjusted_bbox". What's left
        is each object's position relative to the fixed pitch, not the camera --
        which is what view_transformer and speed_and_distance_estimator need.
        """
        cumulative_x, cumulative_y = 0.0, 0.0
        for frame_num in range(len(camera_movement)):
            cumulative_x += camera_movement[frame_num][0]
            cumulative_y += camera_movement[frame_num][1]

            for obj_type in ["players", "ball"]:
                for track_id, track in tracks[obj_type][frame_num].items():
                    bbox = track["bbox"]
                    tracks[obj_type][frame_num][track_id]["adjusted_bbox"] = [
                        bbox[0] - cumulative_x,
                        bbox[1] - cumulative_y,
                        bbox[2] - cumulative_x,
                        bbox[3] - cumulative_y,
                    ]


In [ ]:
%%writefile view_transformer/view_transformer.py
import sys
sys.path.append('/content')

import cv2
import numpy as np

from utils.bbox_utils import get_center_of_bbox, get_foot_position


class ViewTransformer:
    """
    Maps pixel positions from the video into real-world pitch coordinates (meters),
    via a homography computed once from a handful of known reference points -- e.g.
    the pitch's four corners -- whose pixel location you read off a frame, paired
    with their known real-world location on an actual pitch.

    This project's footage is a single, effectively static full-pitch overhead shot
    (the whole pitch is visible in every frame), so ONE homography, computed once,
    covers the entire clip -- unlike a panning/zooming broadcast camera, which would
    need this recomputed per frame or per shot.

    IMPORTANT: pixel_points must be given in order going around the pitch boundary
    (e.g. top-left, top-right, bottom-right, bottom-left) -- not paired diagonally --
    since they're also used to build the polygon that decides whether a given point
    is inside the calibrated pitch region at all.
    """

    def __init__(self, pixel_points, target_points):
        pixel_points = np.array(pixel_points, dtype=np.float32)
        target_points = np.array(target_points, dtype=np.float32)
        if len(pixel_points) < 4 or len(pixel_points) != len(target_points):
            raise ValueError("Need at least 4 matching pixel/target reference point pairs")

        self.pixel_polygon = pixel_points
        homography, _ = cv2.findHomography(pixel_points, target_points)
        if homography is None:
            raise ValueError("Could not compute a homography from the given reference points "
                              "-- check they aren't collinear or duplicated")
        self.perspective_transformer = homography

    def transform_point(self, point):
        """
        point: (x, y) pixel coordinate. Returns the corresponding (x, y) real-world
        pitch coordinate in meters, or None if the point falls outside the reference
        polygon. Extrapolating a homography far outside the region it was calibrated
        on gives meaningless (sometimes wildly wrong) results, so out-of-bounds points
        are refused rather than silently "transformed" into garbage.
        """
        p = (float(point[0]), float(point[1]))
        is_inside = cv2.pointPolygonTest(self.pixel_polygon, p, False) >= 0
        if not is_inside:
            return None

        reshaped = np.array([point], dtype=np.float32).reshape(-1, 1, 2)
        transformed = cv2.perspectiveTransform(reshaped, self.perspective_transformer)
        return transformed.reshape(-1, 2)[0]

    def transform_tracks(self, tracks, pixel_scale=1.0):
        """
        Adds a "position_transformed" key (real-world [x, y] in meters, or None if
        outside the calibrated pitch region) to every player/ball track, using each
        object's camera-motion-adjusted position if camera_movement_estimator has
        already run (falls back to the raw bbox otherwise): foot position for players
        (where they're actually standing on the pitch), center for the ball.

        pixel_scale: multiply the extracted (bbox-derived) position by this factor
        before transforming, WITHOUT touching the stored bbox/adjusted_bbox itself.

        This matters because the homography's reference points (pixel_points passed
        to __init__) are read off a NATIVE-resolution frame (e.g. via get_native_frame),
        but the bboxes stored in `tracks` come from the tracker running on
        read_video()'s DOWNSAMPLED frames (default target_width=1920). If those two
        pixel spaces don't match, every position silently transforms into the wrong
        real-world location -- typically clustering everyone near one corner of the
        pitch instead of spanning it (and, less visibly, scaling every speed/distance
        number by roughly the same downsample ratio). Pass
        pixel_scale = native_frame_width / downsampled_frame_width to correct for it;
        leave it at the default 1.0 when bboxes are already in the same pixel space
        the homography was calibrated in.
        """
        for obj_type in ["players", "ball"]:
            for frame_num, frame_tracks in enumerate(tracks[obj_type]):
                for track_id, track in frame_tracks.items():
                    bbox = track.get("adjusted_bbox", track["bbox"])
                    position = get_foot_position(bbox) if obj_type == "players" else get_center_of_bbox(bbox)
                    scaled_position = (position[0] * pixel_scale, position[1] * pixel_scale)
                    transformed = self.transform_point(scaled_position)
                    tracks[obj_type][frame_num][track_id]["position_transformed"] = (
                        transformed.tolist() if transformed is not None else None
                    )


In [ ]:
%%writefile speed_and_distance_estimator/speed_and_distance_estimator.py
import sys
sys.path.append('/content')

import cv2

from utils.bbox_utils import measure_distance, get_foot_position


class SpeedAndDistanceEstimator:
    """
    Turns each player's real-world (meters) position over time into running distance
    covered (meters) and instantaneous speed (km/h). Computed over a rolling window of
    several frames rather than frame-to-frame -- frame-to-frame position jitter (a few
    pixels of detection noise, which becomes a few centimeters of real-world noise)
    would otherwise translate into wildly unstable speed readings.
    """

    def __init__(self, frame_window=5, fps=25):
        self.frame_window = frame_window
        self.fps = fps

    def add_speed_and_distance(self, tracks):
        total_distance = {}
        n_frames = len(tracks["players"])

        for frame_num in range(0, n_frames, self.frame_window):
            last_frame = min(frame_num + self.frame_window, n_frames - 1)
            if last_frame == frame_num:
                continue

            for track_id, track in tracks["players"][frame_num].items():
                if track_id not in tracks["players"][last_frame]:
                    continue

                start_pos = track.get("position_transformed")
                end_pos = tracks["players"][last_frame][track_id].get("position_transformed")
                if start_pos is None or end_pos is None:
                    continue

                distance_covered = measure_distance(start_pos, end_pos)
                time_elapsed = (last_frame - frame_num) / self.fps
                if time_elapsed <= 0:
                    continue

                speed_kmph = (distance_covered / time_elapsed) * 3.6
                total_distance[track_id] = total_distance.get(track_id, 0.0) + distance_covered

                # +1 so the window's own last frame gets written too -- otherwise the
                # very last frame of the whole clip (where last_frame == n_frames - 1
                # and there's no further window to start from it) would never get a
                # speed/distance value at all. Consecutive windows' ranges touch at
                # exactly one frame (this window's last_frame == the next window's
                # frame_num); that frame simply gets overwritten by the next window's
                # value a moment later, which is harmless.
                for fn in range(frame_num, last_frame + 1):
                    if track_id in tracks["players"][fn]:
                        tracks["players"][fn][track_id]["speed"] = speed_kmph
                        tracks["players"][fn][track_id]["distance"] = total_distance[track_id]

    def draw_frame_speed_and_distance(self, frame, frame_num, tracks):
        """
        Single-frame version of draw_speed_and_distance -- draws directly onto `frame`
        and returns it, for use by the streaming renderer (see
        utils/video_utils.py's render_video_streaming).
        """
        for _, track in tracks["players"][frame_num].items():
            if "speed" not in track:
                continue
            bbox = track.get("adjusted_bbox", track["bbox"])
            x, y = get_foot_position(bbox)
            y += 40
            cv2.putText(frame, f"{track['speed']:.1f} km/h", (int(x), int(y)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 0), 2)
            y += 15
            cv2.putText(frame, f"{track['distance']:.1f} m", (int(x), int(y)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 0), 2)
        return frame

    def draw_speed_and_distance(self, frames, tracks):
        """
        List-based version -- builds and returns a full second copy of the clip. For
        anything more than a couple hundred frames, prefer render_video_streaming()
        with draw_frame_speed_and_distance instead (see utils/video_utils.py). Kept
        here for backward compatibility.
        """
        output = []
        for frame_num, frame in enumerate(frames):
            frame = frame.copy()
            frame = self.draw_frame_speed_and_distance(frame, frame_num, tracks)
            output.append(frame)
        return output


In [ ]:
%%writefile heatmap/heatmap.py
import sys
sys.path.append('/content')

import numpy as np


class PositionHeatmapBuilder:
    """
    Stage 4.1 -- accumulates each player's real-world (meters) position across every
    frame they were tracked, and bins it into a 2D density grid over the pitch --
    "where did this player spend the match." Exported as a JSON-serializable grid
    (raw counts, and a normalized density that sums to 1 over sampled cells), NOT a
    baked image -- a future frontend can render it however it wants (color scale,
    opacity, zoom) instead of being stuck with a fixed picture.

    Deliberately a plain 2D histogram rather than a full kernel-density estimate --
    coarser, but exactly reproducible and trivially testable; a smoother KDE-style
    look is a rendering choice a frontend can add on top of this raw grid (e.g. a
    blur pass) if it wants one, without changing what's actually being measured here.
    """

    def __init__(self, pitch_length_m, pitch_width_m, cell_size_m=2.0):
        self.pitch_length_m = pitch_length_m
        self.pitch_width_m = pitch_width_m
        self.cell_size_m = cell_size_m
        self.n_cols = int(np.ceil(pitch_length_m / cell_size_m))
        self.n_rows = int(np.ceil(pitch_width_m / cell_size_m))

    def _cell_index(self, x, y):
        col = int(x // self.cell_size_m)
        row = int(y // self.cell_size_m)
        col = max(0, min(self.n_cols - 1, col))
        row = max(0, min(self.n_rows - 1, row))
        return row, col

    def build_player_grids(self, tracks):
        """
        tracks: pipeline tracks dict, with `position_transformed` already populated
        by ViewTransformer.transform_tracks().

        Returns {track_id: {"team": 1/2/"referee", "grid": [[raw counts]],
        "n_samples": n}} -- one entry per player who had at least one valid
        real-world position anywhere in the clip. `grid` is n_rows x n_cols of RAW
        counts (not yet normalized), so grids can still be combined/compared before
        normalizing.
        """
        grids = {}
        for player_track in tracks["players"]:
            for track_id, track in player_track.items():
                pos = track.get("position_transformed")
                if pos is None:
                    continue
                if track_id not in grids:
                    grids[track_id] = {
                        "team": track.get("team"),
                        "grid": [[0] * self.n_cols for _ in range(self.n_rows)],
                        "n_samples": 0,
                    }
                row, col = self._cell_index(pos[0], pos[1])
                grids[track_id]["grid"][row][col] += 1
                grids[track_id]["n_samples"] += 1
        return grids

    def to_density_json(self, grids):
        """
        Normalizes each player's raw-count grid to a density (sums to 1 across all
        cells) for JSON export. A player with 0 samples keeps an all-zero grid rather
        than dividing by zero.
        """
        result = {}
        for track_id, entry in grids.items():
            n = entry["n_samples"]
            if n > 0:
                density = [[c / n for c in row] for row in entry["grid"]]
            else:
                density = entry["grid"]
            result[str(track_id)] = {
                "team": entry["team"],
                "density_grid": density,
                "n_samples": n,
                "cell_size_m": self.cell_size_m,
                "pitch_length_m": self.pitch_length_m,
                "pitch_width_m": self.pitch_width_m,
            }
        return result


In [ ]:
%%writefile player_rating/player_rating.py
import sys
sys.path.append('/content')


class PlayerRatingModel:
    """
    Stage 4.2 -- a self-built, explainable 1-10 rating per player per match, from
    stats that are honestly measurable off single-camera 2D tracking: distance
    covered, sprint count, possession involvement, and ball recoveries (a
    proximity/possession-based turnover heuristic). Deliberately NOT presented as
    equivalent to FotMob/Sofascore's proprietary event-based ratings -- this is a
    transparent, from-scratch model built on a much smaller set of honestly-available
    inputs, and its weights are a stated, arguable design choice, not a hidden
    formula. Pass count (only meaningful with Stage 4.3's event detection) is
    intentionally left out here -- 4.3 wasn't built, so it's not silently assumed.

    Weighting rationale (why these weights, not others -- meant to be re-argued, not
    treated as settled):
      - distance_covered (25%): a durable, low-noise work-rate signal -- every
        player's distance is measured the same reliable way (Stage 3.3's running
        total), so it anchors the volume side of the rating.
      - sprint_count (20%): distinguishes genuine high-intensity contribution from a
        player who covers the same ground at a jog -- frames above
        `sprint_speed_kmh` count.
      - possession_involvement (30%): the stat most tied to actually being part of
        the game rather than just running around it -- weighted highest of the four.
      - ball_recoveries (25%): a defensive/disruption signal the pure-movement stats
        above don't capture at all -- without it, a rating would silently reward only
        attacking-minded running and ignore defensive contribution entirely.
    A coach might reasonably reweight these per position (recoveries higher for a
    defensive mid, possession_involvement higher for a playmaker) -- this model uses
    one fixed set of weights for every player, which is itself a stated
    simplification worth naming, not hiding.
    """

    DEFAULT_WEIGHTS = {
        "distance_covered": 0.25,
        "sprint_count": 0.20,
        "possession_involvement": 0.30,
        "ball_recoveries": 0.25,
    }

    def __init__(self, weights=None, sprint_speed_kmh=20.0):
        self.weights = dict(weights) if weights is not None else dict(self.DEFAULT_WEIGHTS)
        total = sum(self.weights.values())
        if abs(total - 1.0) > 1e-6:
            raise ValueError(f"Weights must sum to 1.0, got {total}")
        self.sprint_speed_kmh = sprint_speed_kmh

    def compute_raw_stats(self, tracks, team_ball_control):
        """
        tracks: pipeline tracks dict with `speed`, `distance`, `has_ball`, and `team`
        already populated per player per frame (Stages 2 and 3's own outputs).
        team_ball_control: per-frame sequence of which team (1/2, or 0) currently has
        the ball.

        Returns {track_id: {"team", "distance_covered", "sprint_count",
        "possession_involvement", "ball_recoveries"}}:
          - distance_covered: that player's own final cumulative `distance` (Stage
            3.3 already tracks a running total per player) -- the max value seen,
            since it's monotonically non-decreasing.
          - sprint_count: number of frames where that player's `speed` exceeded
            `sprint_speed_kmh`.
          - possession_involvement: number of frames flagged `has_ball` for that
            player (Stage 2.5's ball-assignment signal) -- literal touches; a simple,
            honest proxy for "involved in the game," not a full pass/event count.
          - ball_recoveries: number of frames where team_ball_control changed TO that
            player's team on that exact frame AND that specific player is the one
            holding the ball that frame -- i.e. the player whose touch coincided with
            a turnover in their team's favor. A simple heuristic, explicitly not a
            modeled "tackle" or "interception" -- the same limitation the Build Plan
            itself names for this stat.
        """
        stats = {}

        def _ensure(track_id, team):
            if track_id not in stats:
                stats[track_id] = {
                    "team": team, "distance_covered": 0.0, "sprint_count": 0,
                    "possession_involvement": 0, "ball_recoveries": 0,
                }

        for frame_num, player_track in enumerate(tracks["players"]):
            prev_team = team_ball_control[frame_num - 1] if frame_num > 0 else 0
            cur_team = team_ball_control[frame_num] if frame_num < len(team_ball_control) else 0
            turnover_to = cur_team if (cur_team in (1, 2) and cur_team != prev_team) else None

            for track_id, track in player_track.items():
                team = track.get("team")
                if team not in (1, 2):
                    continue
                _ensure(track_id, team)

                dist = track.get("distance")
                if dist is not None:
                    stats[track_id]["distance_covered"] = max(stats[track_id]["distance_covered"], dist)

                speed = track.get("speed")
                if speed is not None and speed > self.sprint_speed_kmh:
                    stats[track_id]["sprint_count"] += 1

                if track.get("has_ball"):
                    stats[track_id]["possession_involvement"] += 1
                    if turnover_to == team:
                        stats[track_id]["ball_recoveries"] += 1

        return stats

    @staticmethod
    def _normalize(values):
        """
        0-1 min-max normalizes a {id: value} dict. When every value is equal
        (including just one player) there's no spread to rank them apart by, so
        everyone normalizes to 1.0 rather than an arbitrary 0.0 -- a player isn't
        penalized just for being alone in, or tied within, the comparison set.
        """
        if not values:
            return {}
        lo, hi = min(values.values()), max(values.values())
        if hi - lo < 1e-9:
            return {k: 1.0 for k in values}
        return {k: (v - lo) / (hi - lo) for k, v in values.items()}

    def rate_players(self, tracks, team_ball_control):
        """
        Returns {track_id: {"team", "rating_1_10", "raw_stats", "normalized_stats"}}.
        Each stat is min-max normalized 0-1 ACROSS ALL PLAYERS IN THIS MATCH (per the
        Build Plan's own spec), weighted, summed, then mapped from [0,1] to [1,10].
        """
        raw_stats = self.compute_raw_stats(tracks, team_ball_control)

        normalized_by_stat = {}
        for stat_name in self.weights:
            values = {tid: s[stat_name] for tid, s in raw_stats.items()}
            normalized_by_stat[stat_name] = self._normalize(values)

        results = {}
        for track_id, raw in raw_stats.items():
            normalized = {stat: normalized_by_stat[stat][track_id] for stat in self.weights}
            score_0_1 = sum(self.weights[stat] * normalized[stat] for stat in self.weights)
            rating = 1.0 + score_0_1 * 9.0
            results[track_id] = {
                "team": raw["team"],
                "rating_1_10": round(rating, 2),
                "raw_stats": raw,
                "normalized_stats": {k: round(v, 3) for k, v in normalized.items()},
            }
        return results


## Run detection, tracking, team assignment, possession

Same logic as Stages 2–3, running now on the broadcast model and clip instead of topview.

In [ ]:
import sys
sys.path.append('/content')
import numpy as np
import pandas as pd
import cv2

from utils.video_utils import read_video, save_video, get_native_frame, render_video_streaming
from trackers.tracker import Tracker
from team_assigner.team_assigner import TeamAssigner
from player_ball_assigner.player_ball_assigner import PlayerBallAssigner
from camera_movement_estimator.camera_movement_estimator import CameraMovementEstimator
from view_transformer.view_transformer import ViewTransformer
from speed_and_distance_estimator.speed_and_distance_estimator import SpeedAndDistanceEstimator
from heatmap.heatmap import PositionHeatmapBuilder
from player_rating.player_rating import PlayerRatingModel

video_frames = read_video(clip_path)
print(f"Loaded {len(video_frames)} frames")

tracker = Tracker(model_path)
tracks = tracker.get_object_tracks(video_frames)
print("Detection + tracking done")

In [ ]:
ball_positions = [x.get(1, {}).get("bbox", [np.nan] * 4) for x in tracks["ball"]]
df_ball = pd.DataFrame(ball_positions, columns=["x1", "y1", "x2", "y2"])
df_ball = df_ball.interpolate().bfill()
tracks["ball"] = [{1: {"bbox": row}} for row in df_ball.to_numpy().tolist()]
print("Ball positions interpolated")

In [ ]:
native_frame0 = get_native_frame(clip_path, 0)
scale = native_frame0.shape[1] / video_frames[0].shape[1]

def _scale_bbox(bbox, s):
    return [c * s for c in bbox]

team_assigner = TeamAssigner()

n_frames = len(tracks["players"])
sample_frame_nums = sorted(set(min(n_frames - 1, int(n_frames * f)) for f in [0.0, 0.2, 0.4, 0.6, 0.8]))

per_player_samples = {}
for fn in sample_frame_nums:
    native = get_native_frame(clip_path, fn)
    for player_id, track in tracks["players"][fn].items():
        color = team_assigner.get_player_color(native, _scale_bbox(track["bbox"], scale))
        per_player_samples.setdefault(player_id, []).append(color)

aggregate_colors = {tid: np.median(np.array(colors), axis=0) for tid, colors in per_player_samples.items()}
team_assigner.assign_team_colors_from_samples(aggregate_colors)

for frame_num, player_track in enumerate(tracks["players"]):
    native_frame = None
    for player_id, track in player_track.items():
        if player_id in team_assigner.player_team_dict:
            team = team_assigner.player_team_dict[player_id]
        else:
            if native_frame is None:
                native_frame = get_native_frame(clip_path, frame_num)
            team = team_assigner.get_player_team(native_frame, _scale_bbox(track["bbox"], scale), player_id)

        tracks["players"][frame_num][player_id]["team"] = team
        if team == "referee":
            tracks["players"][frame_num][player_id]["team_color"] = (255, 255, 255)
        else:
            tracks["players"][frame_num][player_id]["team_color"] = tuple(
                int(c) for c in team_assigner.team_colors[team]
            )

print("Team colors (BGR):", team_assigner.team_colors)
referee_count = sum(1 for p in tracks["players"][0].values() if p.get("team") == "referee")
print(f"Frame 0: {len(tracks['players'][0])} people detected, {referee_count} flagged as referee")
print("If this looks wrong on broadcast footage: this method was originally tuned for topview footage's")
print("grass contamination (grass-hue exclusion). Broadcast crops have much less grass in the top-half")
print("sample region to begin with, so it should still work, but re-check frame 0's team-colored output")
print("before trusting the rest of the pipeline -- same as every previous stage.")

In [ ]:
ball_assigner = PlayerBallAssigner()
team_ball_control = []
for frame_num, player_track in enumerate(tracks["players"]):
    ball_bbox = tracks["ball"][frame_num][1]["bbox"]
    assigned_player = ball_assigner.assign_ball_to_player(player_track, ball_bbox)

    if assigned_player != -1:
        tracks["players"][frame_num][assigned_player]["has_ball"] = True
        assigned_team = tracks["players"][frame_num][assigned_player].get("team")
        if assigned_team in (1, 2):
            team_ball_control.append(assigned_team)
        else:
            team_ball_control.append(team_ball_control[-1] if team_ball_control else 0)
    else:
        team_ball_control.append(team_ball_control[-1] if team_ball_control else 0)

team_ball_control = np.array(team_ball_control)
print("Possession computed")

## Camera motion compensation

Broadcast footage typically pans and zooms to follow play — unlike the topview notebook's static drone shot, this step should now actually detect real camera movement, not just confirm ~0.

In [ ]:
camera_estimator = CameraMovementEstimator(video_frames[0])
camera_movement = camera_estimator.get_camera_movement(video_frames)
camera_estimator.adjust_positions_to_tracks(tracks, camera_movement)

max_movement = max(abs(dx) + abs(dy) for dx, dy in camera_movement)
print(f"Max per-frame camera movement detected: {max_movement:.1f}px "
      f"({'camera is panning/zooming, as expected for broadcast footage' if max_movement >= 5 else 'looks static -- unusual for broadcast, worth a visual sanity check'})")

## Calibrate the pixel-to-pitch homography for THIS clip

Same manual step as Stage 3, but it has to be redone here — this is a different camera angle from the topview clip, so its 4 pitch-corner pixels are different numbers. The cell below saves and displays this clip's first frame with a coordinate grid overlaid. Read off the approximate **pixel (x, y)** of the pitch's four corners (or another set of 4 known points, e.g. the penalty box corners, if all 4 pitch corners aren't visible in a broadcast camera's framing), going around the boundary in order — top-left, top-right, bottom-right, bottom-left. Then edit `PIXEL_CORNERS` in the next cell with what you read off (it's pre-filled with placeholder values that will NOT be correct for your clip).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

fig, ax = plt.subplots(figsize=(16, 9))
ax.imshow(cv2.cvtColor(native_frame0, cv2.COLOR_BGR2RGB))
ax.xaxis.set_major_locator(ticker.MultipleLocator(200))
ax.yaxis.set_major_locator(ticker.MultipleLocator(200))
ax.grid(True, color='yellow', alpha=0.5, linewidth=0.5)
ax.set_title(f"Native frame 0 ({native_frame0.shape[1]}x{native_frame0.shape[0]}px) -- read off 4 known pitch points' pixel coordinates")
plt.savefig('frame0_grid.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Frame size: {native_frame0.shape[1]} wide x {native_frame0.shape[0]} tall")

In [ ]:
# PLACEHOLDER -- edit these 4 points after looking at the grid image above. Go around
# the boundary in order (top-left, top-right, bottom-right, bottom-left); don't skip
# diagonally, or the homography will silently come out wrong.
#
# If a broadcast camera's framing doesn't show all 4 real pitch corners at once, use any
# 4 known points instead (e.g. the visible penalty box's 4 corners are 40.3m x 16.5m per
# FIFA standard) -- just make sure REAL_CORNERS below matches whichever 4 points you pick.
PIXEL_CORNERS = [
    [100, 100],    # top-left -- REPLACE with your reading
    [1800, 100],   # top-right -- REPLACE
    [1800, 1000],  # bottom-right -- REPLACE
    [100, 1000],   # bottom-left -- REPLACE
]

# Real pitch dimensions in meters -- FIFA standard is 105 x 68. Change if this pitch's
# actual dimensions are known, or if PIXEL_CORNERS above are the penalty box instead
# (40.3 x 16.5) -- REAL_CORNERS must describe whatever 4 points PIXEL_CORNERS actually is.
PITCH_LENGTH_M = 105
PITCH_WIDTH_M = 68

REAL_CORNERS = [
    [0, 0],
    [PITCH_LENGTH_M, 0],
    [PITCH_LENGTH_M, PITCH_WIDTH_M],
    [0, PITCH_WIDTH_M],
]

view_transformer = ViewTransformer(PIXEL_CORNERS, REAL_CORNERS)
# Same pixel_scale correction as Stage 3: PIXEL_CORNERS are read off the NATIVE-resolution
# frame, but tracked bboxes are in read_video()'s DOWNSAMPLED pixel space -- without this,
# every position transforms into the wrong real-world spot. See view_transformer.py's own
# docstring for the full explanation.
view_transformer.transform_tracks(tracks, pixel_scale=scale)

in_bounds = sum(
    1 for p in tracks["players"][0].values() if p.get("position_transformed") is not None
)
print(f"Frame 0: {in_bounds}/{len(tracks['players'][0])} players landed inside the calibrated pitch region")
print("If that number is 0 or very low, PIXEL_CORNERS is still the placeholder above (or misread) --")
print("go back, read the grid image, and edit PIXEL_CORNERS before trusting anything below this cell.")

In [ ]:
speed_estimator = SpeedAndDistanceEstimator(frame_window=5, fps=25)  # adjust fps if this clip isn't 25fps
speed_estimator.add_speed_and_distance(tracks)
print("Speed and distance computed")

**Memory note:** this renders straight to the video file frame by frame (`render_video_streaming`) instead of building two extra full copies of the clip in Python lists first. The old two-pass approach (`draw_annotations()` then `draw_speed_and_distance()`, each returning a full new list) can need 3x+ the clip's frame data in RAM at once — on a clip this length that's enough to crash a free-tier Colab session with "used all available RAM" right at this step, even though detection/tracking already succeeded. This does the same drawing in one streaming pass instead.

In [ ]:
render_video_streaming(
    video_frames, 'stage4_broadcast_output.mp4',
    draw_fns=[
        lambda f, n: tracker.draw_frame_annotations(f, n, tracks, team_ball_control),
        lambda f, n: speed_estimator.draw_frame_speed_and_distance(f, n, tracks),
    ],
    fps=25,
)
print("Saved stage4_broadcast_output.mp4 -- sanity-check this BEFORE trusting the heatmaps/ratings below:")
print("do the ellipses track real players, are team colors right, do speeds look plausible (not 0 or 300 km/h)?")
files.download('stage4_broadcast_output.mp4')

# video_frames was just mutated in place by the drawing above and isn't needed again --
# free it now, before the heatmap/rating cells below, rather than letting it sit in RAM
# for the rest of the session.
import gc
del video_frames
gc.collect()
print("Freed the raw video frames from memory (not needed again below).")

## Stage 4.1: position heatmaps

Bins each player's real-world position into a density grid over the pitch -- "where did this player spend the match." Exported as JSON (a raw grid, not a picture) so a future dashboard can render it however it wants.

In [ ]:
heatmap_builder = PositionHeatmapBuilder(pitch_length_m=PITCH_LENGTH_M, pitch_width_m=PITCH_WIDTH_M, cell_size_m=2.0)
player_grids = heatmap_builder.build_player_grids(tracks)
heatmaps_json = heatmap_builder.to_density_json(player_grids)

print(f"Built heatmaps for {len(heatmaps_json)} players.")
for tid, entry in list(heatmaps_json.items())[:5]:
    print(f"  player {tid} (team {entry['team']}): {entry['n_samples']} samples")

import json as _json
with open('heatmaps.json', 'w') as f:
    _json.dump(heatmaps_json, f)
files.download('heatmaps.json')
print("Saved heatmaps.json")

In [ ]:
# Quick visual sanity check -- NOT the final frontend rendering (that's Stage 5), just
# confirming the density grids look like plausible player movement, not noise.
top_players = sorted(heatmaps_json.items(), key=lambda kv: kv[1]["n_samples"], reverse=True)[:4]

fig, axes = plt.subplots(1, len(top_players), figsize=(5 * len(top_players), 4.5))
if len(top_players) == 1:
    axes = [axes]
for ax, (tid, entry) in zip(axes, top_players):
    grid = np.array(entry["density_grid"])
    ax.imshow(grid, origin='lower', extent=[0, PITCH_LENGTH_M, 0, PITCH_WIDTH_M],
               cmap='hot', aspect='auto')
    ax.set_title(f"Player {tid} (team {entry['team']}, {entry['n_samples']} samples)")
    ax.set_xlabel('pitch length (m)')
plt.tight_layout()
plt.savefig('heatmaps_preview.png', dpi=100)
plt.show()

## Stage 4.2: custom player-rating model

An explainable 1-10 rating per player from distance covered, sprint count, possession involvement, and ball recoveries. See `player_rating.py`'s own docstring for the full weighting rationale -- worth reading before treating any single number here as final.

In [ ]:
rating_model = PlayerRatingModel()  # default weights: distance 25%, sprints 20%, possession 30%, recoveries 25%
ratings = rating_model.rate_players(tracks, team_ball_control.tolist())

ranked = sorted(ratings.items(), key=lambda kv: kv[1]["rating_1_10"], reverse=True)
print(f"{'Player':>8}  {'Team':>4}  {'Rating':>6}  {'Dist(m)':>8}  {'Sprints':>7}  {'Poss.frames':>11}  {'Recoveries':>10}")
for tid, r in ranked:
    s = r["raw_stats"]
    print(f"{tid:>8}  {r['team']:>4}  {r['rating_1_10']:>6.2f}  {s['distance_covered']:>8.1f}  "
          f"{s['sprint_count']:>7}  {s['possession_involvement']:>11}  {s['ball_recoveries']:>10}")

import json as _json
with open('ratings.json', 'w') as f:
    _json.dump(ratings, f, default=lambda o: int(o) if hasattr(o, 'item') else o)
files.download('ratings.json')
print("\nSaved ratings.json")
print("\nRemember: normalization is relative to THIS match's players only -- a rating of 7.0 here means")
print("'7th-percentile-equivalent among the ~20ish players in this clip,' not a fixed universal scale.")
print("A short clip with few players also means these numbers are noisier than a full 90-minute match would give.")

## Bring the code home

Download the actual pipeline files to drop into your local project folder.

In [ ]:
import zipfile, os

with zipfile.ZipFile('stage4_analytics_modules.zip', 'w') as z:
    for folder in ['utils', 'trackers', 'team_assigner', 'player_ball_assigner',
                    'camera_movement_estimator', 'view_transformer', 'speed_and_distance_estimator',
                    'heatmap', 'player_rating']:
        for root, _, filenames in os.walk(folder):
            for fn in filenames:
                z.write(os.path.join(root, fn))
files.download('stage4_analytics_modules.zip')
print("Downloaded stage4_analytics_modules.zip — unzip into your local project folder root.")

### Done for now

You should have downloaded `stage4_broadcast_output.mp4` (the annotated tracking video — check this FIRST), `heatmaps.json`, `ratings.json`, `heatmaps_preview.png`, and `stage4_analytics_modules.zip`.

**Not built here, and why:** 4.3 (shot detection + a simplified xG model) is the Build Plan's own optional stretch phase — it needs ball-speed-spike shot detection, a goal-box trajectory filter, AND a public open dataset (StatsBomb open data) to train a logistic regression on distance/angle-to-goal. That's a genuinely separate, heavier piece of work with an external data dependency this pipeline doesn't have yet, not something to fake with a placeholder. Full pass/dribble event detection is explicitly called out in the Build Plan as noisier on single-camera 2D tracking and named as future work, not a Stage 4 requirement — so `possession_involvement` (literal ball touches) stands in for it here, not a pass count.

**Known limitations worth stating plainly, same spirit as every prior stage:**
- The rating model's weights (25/20/30/25) are a stated, arguable starting point — re-justify or re-tune them once you can watch this against a match you know well.
- `ball_recoveries` is a simple possession-based turnover heuristic, not a modeled tackle/interception — it credits whoever's holding the ball the instant `team_ball_control` flips, nothing more.
- Ratings only compare players within THIS match — there's no cross-match baseline yet (that would need Stage 5's multi-match library).
- If this clip's footage isn't 25fps, fix `fps=` in the `SpeedAndDistanceEstimator(...)` call above before trusting speed/distance numbers, same as Stage 3.